# Transformer with PyTorch and Hugging Face

This notebook solves the same text classification task in two ways: a small Transformer encoder trained from zero with PyTorch's `nn.TransformerEncoder`, and a pre-trained DistilBERT fine-tuned with Hugging Face `transformers`. See "Transformer from Scratch" for an implementation of each component.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/08_transformers'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
%pip install --quiet datasets

## Dataset

[Rotten Tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes): movie review sentences labeled as negative (0) or positive (1). Both models use the same tokenizer.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

checkpoint = "distilbert-base-uncased"
max_length = 64

dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized = dataset.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=max_length), batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tokenizer)

print(dataset)
print(dataset["train"][0])

## PyTorch: Transformer trained from zero

The model embeds the tokens, adds positional information, and passes the sequence through `nn.TransformerEncoder`. Each encoder layer already contains multi-head self-attention and the feed-forward block. The sentence representation is the average of the token outputs, ignoring padding.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2, num_classes=2):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = nn.Embedding(max_length, d_model)
        layer = nn.TransformerEncoderLayer(d_model, num_heads, dim_feedforward=4 * d_model, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.embedding(input_ids) * math.sqrt(self.d_model) + self.position(positions)
        x = self.encoder(x, src_key_padding_mask=attention_mask == 0)
        mask = attention_mask.unsqueeze(-1).float()
        return self.classifier((x * mask).sum(dim=1) / mask.sum(dim=1))

In [ ]:
num_epochs = 5
train_loader = DataLoader(tokenized["train"], batch_size=64, shuffle=True, collate_fn=collator)
test_loader = DataLoader(tokenized["test"], batch_size=64, collate_fn=collator)

model = TransformerClassifier(tokenizer.vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"])
        correct += (logits.argmax(dim=1) == batch["labels"]).sum().item()
        total += len(batch["labels"])
    return correct / total


loss_history = []
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = F.cross_entropy(model(batch["input_ids"], batch["attention_mask"]), batch["labels"])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}, Test accuracy: {evaluate(model, test_loader):.3f}")

pytorch_accuracy = evaluate(model, test_loader)

In [ ]:
plt.plot(loss_history, label="Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Hugging Face: fine-tuning a pre-trained Transformer

DistilBERT was pre-trained on a large text corpus. `AutoModelForSequenceClassification` loads it and adds a new classification head, and `Trainer` takes care of the training loop.

In [ ]:
hf_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    return {"accuracy": (np.argmax(logits, axis=1) == labels).mean()}


training_args = TrainingArguments(
    output_dir="distilbert-rotten-tomatoes",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=hf_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
hf_accuracy = trainer.evaluate()["eval_accuracy"]

## Test

In [ ]:
print(f"Transformer trained from zero: {pytorch_accuracy:.3f}")
print(f"Pre-trained DistilBERT: {hf_accuracy:.3f}")

In [ ]:
sentences = ["A beautiful and moving film.", "Two hours of my life I will never get back."]
inputs = tokenizer(sentences, padding=True, return_tensors="pt").to(hf_model.device)

with torch.no_grad():
    predictions = hf_model(**inputs).logits.argmax(dim=1)

for sentence, prediction in zip(sentences, predictions):
    print(f"{['negative', 'positive'][prediction]}: {sentence}")